# LSTM Numerical Example — Implemented from Scratch
**Based on:** lecture3-_LSTM_numerical_example.pdf  
**Course:** Deep Learning — A. Prof. Noha El-Attar  

**Problem:** Given the sequence `[1, 2, 3]`, predict the next value `4`  
- Input dimension = 1  
- Hidden state dimension = 1  
- Expected prediction ≈ **3.8** (close to 4)

In [1]:
import numpy as np

# Activation functions
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def tanh(x):
    return np.tanh(x)

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Step 1: Initialize Parameters

# Forget gate weights — decide what to forget from memory
Wxf = np.array([[0.1]])
Whf = np.array([[0.2]])
bf  = np.array([[0.3]])

# Input gate weights — decide what new info to store
# I_t = sigmoid(x_t * Wxi + H_{t-1} * Whi + bi)
Wxi = np.array([[0.4]])
Whi = np.array([[0.5]])
bi  = np.array([[0.6]])

# Candidate cell state weights — new candidate values
Wxc = np.array([[0.7]])
Whc = np.array([[0.8]])
bc  = np.array([[0.9]])

# Output gate weights — decide what to output
Wxo = np.array([[0.1]])
Who = np.array([[0.2]])
bo  = np.array([[0.3]])

# Linear output layer
Wy = np.array([[6.044]])
by = np.array([[0.0]])

# Initial states
H0 = np.array([[0.0]])   # initial hidden state (short memory)
C0 = np.array([[0.0]])   # initial cell state   (long memory)

# Input sequence — predict the 4th value
inputs = [1.0, 2.0, 3.0]

print("Parameters initialized.")
print(f"Forget gate  : Wxf={Wxf.item()}, Whf={Whf.item()}, bf={bf.item()}")
print(f"Input gate   : Wxi={Wxi.item()}, Whi={Whi.item()}, bi={bi.item()}")
print(f"Candidate    : Wxc={Wxc.item()}, Whc={Whc.item()}, bc={bc.item()}")
print(f"Output gate  : Wxo={Wxo.item()}, Who={Who.item()}, bo={bo.item()}")
print(f"Output layer : Wy={Wy.item()},  by={by.item()}")
print(f"Initial H0={H0.item()},  C0={C0.item()}")

Parameters initialized.
Forget gate  : Wxf=0.1, Whf=0.2, bf=0.3
Input gate   : Wxi=0.4, Whi=0.5, bi=0.6
Candidate    : Wxc=0.7, Whc=0.8, bc=0.9
Output gate  : Wxo=0.1, Who=0.2, bo=0.3
Output layer : Wy=6.044,  by=0.0
Initial H0=0.0,  C0=0.0


## Step 2: LSTM Forward Pass

Each time step runs **6 operations**:

| Step | Gate | Formula |
|------|------|---------|
| 1 | Forget gate | `F_t = sigmoid(x·Wxf + H·Whf + bf)` |
| 2 | Input gate  | `I_t = sigmoid(x·Wxi + H·Whi + bi)` |
| 3 | Candidate   | `C_tilde = tanh(x·Wxc + H·Whc + bc)` |
| 4 | Cell update | `C_t = F_t * C_{t-1} + I_t * C_tilde` |
| 5 | Output gate | `O_t = sigmoid(x·Wxo + H·Who + bo)` |
| 6 | Hidden state| `H_t = O_t * tanh(C_t)` |

In [3]:
def lstm_step(x_t, H_prev, C_prev):
    """
    One LSTM time step.
    Inputs : x_t     — current input scalar
             H_prev  — previous hidden state (short memory)
             C_prev  — previous cell state   (long memory)
    Outputs: H_t, C_t, gate_values dict
    """

    # Step 1: Forget gate — what to forget from the previous cell state
    F_t = sigmoid(x_t * Wxf + H_prev * Whf + bf)

    # Step 2: Input gate — what new information to store
    I_t = sigmoid(x_t * Wxi + H_prev * Whi + bi)

    # Step 3: Candidate cell state — new candidate values to add
    C_tilde = tanh(x_t * Wxc + H_prev * Whc + bc)

    # Step 4: Cell state update — combine forget and input
    C_t = F_t * C_prev + I_t * C_tilde

    # Step 5: Output gate — what part of cell state to output
    O_t = sigmoid(x_t * Wxo + H_prev * Who + bo)

    # Step 6: Hidden state update (short memory / output)
    H_t = O_t * tanh(C_t)

    gate_values = {
        'Forget gate  F_t' : F_t.item(),
        'Input gate   I_t' : I_t.item(),
        'Candidate C_tilde' : C_tilde.item(),
        'Cell state   C_t' : C_t.item(),
        'Output gate  O_t' : O_t.item(),
        'Hidden state H_t' : H_t.item(),
    }
    return H_t, C_t, gate_values

print("lstm_step() function defined successfully.")

lstm_step() function defined successfully.


In [4]:
# ── Run Through All Time Steps ──────────────────────────────────
print("=" * 55)
print("   LSTM Numerical Example --- From Scratch (Python)")
print("=" * 55)
print(f"\nInput sequence : {inputs}  -->  Predict next value")
print(f"Initial hidden state H0 = {H0.item()}")
print(f"Initial cell  state  C0 = {C0.item()}")
print()

H = H0.copy()
C = C0.copy()

for t, x_val in enumerate(inputs, start=1):
    x_t = np.array([[x_val]])
    H, C, gates = lstm_step(x_t, H, C)

    print(f"--- Time Step t={t},  Input x={x_val} ---")
    for name, val in gates.items():
        print(f"   {name} = {val:.4f}")
    print()

   LSTM Numerical Example --- From Scratch (Python)

Input sequence : [1.0, 2.0, 3.0]  -->  Predict next value
Initial hidden state H0 = 0.0
Initial cell  state  C0 = 0.0

--- Time Step t=1,  Input x=1.0 ---
   Forget gate  F_t = 0.5987
   Input gate   I_t = 0.7311
   Candidate C_tilde = 0.9217
   Cell state   C_t = 0.6738
   Output gate  O_t = 0.5987
   Hidden state H_t = 0.3517

--- Time Step t=2,  Input x=2.0 ---
   Forget gate  F_t = 0.6388
   Input gate   I_t = 0.8286
   Candidate C_tilde = 0.9886
   Cell state   C_t = 1.2496
   Output gate  O_t = 0.6388
   Hidden state H_t = 0.5419

--- Time Step t=3,  Input x=3.0 ---
   Forget gate  F_t = 0.6700
   Input gate   I_t = 0.8880
   Candidate C_tilde = 0.9979
   Cell state   C_t = 1.7235
   Output gate  O_t = 0.6700
   Hidden state H_t = 0.6287



## Step 3: Predict the Next Value

Apply a **linear output layer**




In [8]:
# Step 3: Predict the next value
# y_hat = H_final * Wy + by
y_hat = float(H.item() * Wy.item() + by.item())

print("=" * 55)
print(f"  Final hidden state  H3  = {H.item():.4f}")
print(f"  Output weight       Wy  = {Wy.item()}")
print(f"  Output bias         by  = {by.item()}")
print()
print(f"  y_hat = H3 * Wy + by")
print(f"        = {H.item():.4f} * {Wy.item()} + {by.item()}")
print(f"        = {y_hat:.4f}")
print()
print(f"  Actual next value        = 4.0")
print(f"  Prediction               = {y_hat:.4f}")
print(f"  --> Close to 4")
print("=" * 55)

  Final hidden state  H3  = 0.6287
  Output weight       Wy  = 6.044
  Output bias         by  = 0.0

  y_hat = H3 * Wy + by
        = 0.6287 * 6.044 + 0.0
        = 3.7998

  Actual next value        = 4.0
  Prediction               = 3.7998
  --> Close to 4


## Summary Table

All gate values for each time step:

In [9]:
# Summary Table — all time steps
print(f"{'t':>3}  {'x':>5}  {'F_t':>8}  {'I_t':>8}  {'C_tilde':>10}  {'C_t':>8}  {'O_t':>8}  {'H_t':>8}")
print("-" * 72)

H = H0.copy()
C = C0.copy()

for t, x_val in enumerate(inputs, start=1):
    x_t = np.array([[x_val]])
    H, C, g = lstm_step(x_t, H, C)
    print(f"{t:>3}  {x_val:>5.1f}  "
          f"{g['Forget gate  F_t']:>8.4f}  "
          f"{g['Input gate   I_t']:>8.4f}  "
          f"{g['Candidate C_tilde']:>10.4f}  "
          f"{g['Cell state   C_t']:>8.4f}  "
          f"{g['Output gate  O_t']:>8.4f}  "
          f"{g['Hidden state H_t']:>8.4f}")

print("-" * 72)
y_hat_final = float(H.item() * Wy.item() + by.item())
print(f"\n  Final prediction  y_hat = {y_hat_final:.4f}  ")
print(f"  Actual value             = 4.0")
print(f"  --> The LSTM predicts {y_hat_final:.2f}, which is close to 4 ")

  t      x       F_t       I_t     C_tilde       C_t       O_t       H_t
------------------------------------------------------------------------
  1    1.0    0.5987    0.7311      0.9217    0.6738    0.5987    0.3517
  2    2.0    0.6388    0.8286      0.9886    1.2496    0.6388    0.5419
  3    3.0    0.6700    0.8880      0.9979    1.7235    0.6700    0.6287
------------------------------------------------------------------------

  Final prediction  y_hat = 3.7998  
  Actual value             = 4.0
  --> The LSTM predicts 3.80, which is close to 4 
